In [1]:
import ast
import numpy as np
import pandas as pd
from dateutil import parser as dateparser
from sentence_transformers import SentenceTransformer

raw = pd.read_csv("../data/raw/resume_data_for_ranking.csv")
res = pd.read_parquet("../data/processed/canonical_resumes_current.parquet").reset_index(drop=True)
job = pd.read_parquet("../data/processed/canonical_jobs_current.parquet").reset_index(drop=True)

assert len(raw) == len(res) == len(job)
print(raw.shape, res.shape, job.shape)

(9544, 35) (9544, 6) (9544, 7)


In [2]:
def join_list_col(x):
    try:
        items = ast.literal_eval(x)
        return " ".join(str(i) for i in items if str(i).strip() and str(i).lower() != "nan")
    except Exception:
        return ""

res["education_text"] = (
    (raw["degree_names"].apply(join_list_col) + " " + raw["major_field_of_studies"].apply(join_list_col))
    .str.strip()
    .replace({"": np.nan})
)

def try_parse_date(s):
    s = str(s).strip()
    if s.lower() in ("till date", "present", "current"):
        return pd.Timestamp.now()
    try:
        return dateparser.parse(s, default=pd.Timestamp(2000, 1, 1))
    except Exception:
        return None

def parse_date_list(x):
    try:
        return [try_parse_date(s) for s in ast.literal_eval(x)]
    except Exception:
        return []

def candidate_experience_years(start_str, end_str, max_plausible=45):
    starts = [d for d in (parse_date_list(start_str) if pd.notna(start_str) else []) if d is not None]
    ends = [d for d in (parse_date_list(end_str) if pd.notna(end_str) else []) if d is not None]
    if not starts or not ends:
        return np.nan
    years = max(0, (max(ends) - min(starts)).days / 365.25)
    return years if years <= max_plausible else np.nan

res["candidate_experience_years"] = [
    candidate_experience_years(s, e) for s, e in zip(raw["start_dates"], raw["end_dates"])
]
job["experience_required_years"] = pd.to_numeric(job["experience_required_years"], errors="coerce")

print(res["education_text"].notna().mean(), res["candidate_experience_years"].notna().mean())

0.9911986588432523 0.8914501257334451


In [3]:
def as_list(x):
    if isinstance(x, (list, tuple, np.ndarray)):
        return [str(s) for s in x]
    return []

In [4]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
# Note: this model truncates at ~256 tokens; longer texts are truncated.

def embed_unique(texts: pd.Series) -> np.ndarray:
    """Encode each unique text once. Return a NaN row if the text is empty."""
    clean = texts.fillna("").astype(str).str.strip()
    uniques = clean[clean != ""].unique().tolist()
    vecs = embedder.encode(uniques, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    lookup = dict(zip(uniques, vecs))
    out = np.full((len(clean), vecs.shape[1]), np.nan, dtype=np.float32)
    for i, t in enumerate(clean):
        if t:
            out[i] = lookup[t]
    return out

def rowwise_cosine(a, b):
    # normalized vectors -> dot product equals cosine similarity; NaNs propagate
    return np.einsum("ij,ij->i", a, b)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

D:\M1\IA\AI_project\ai_model\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\erb\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
feat = pd.DataFrame(index=raw.index)

feat["semantic_similarity"] = rowwise_cosine(
    embed_unique(res["raw_full_text"]), embed_unique(job["raw_full_text"])
)
feat["objective_responsibilities_similarity"] = rowwise_cosine(
    embed_unique(res["candidate_summary"]), embed_unique(job["responsibilities_text"])
)
feat["education_similarity"] = rowwise_cosine(
    embed_unique(res["education_text"]), embed_unique(job["education_requirements_text"])
)

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [6]:
res_skills = res["skills_list"].apply(as_list)
job_skills = job["required_skills_list"].apply(as_list)

def coverage(c, r):
    return len(set(c) & set(r)) / len(set(r)) if c and r else np.nan

def jaccard(c, r):
    return len(set(c) & set(r)) / len(set(c) | set(r)) if c and r else np.nan

feat["skills_coverage"] = [coverage(c, r) for c, r in zip(res_skills, job_skills)]
feat["skills_jaccard"] = [jaccard(c, r) for c, r in zip(res_skills, job_skills)]

# Soft: embed each distinct skill only once
all_skills = sorted({s for l in res_skills for s in l} | {s for l in job_skills for s in l})
skill_vecs = embedder.encode(all_skills, batch_size=128, normalize_embeddings=True, show_progress_bar=True)
skill_idx = {s: i for i, s in enumerate(all_skills)}

def soft_coverage(c, r):
    if not c or not r:
        return np.nan
    C = skill_vecs[[skill_idx[s] for s in c]]
    R = skill_vecs[[skill_idx[s] for s in r]]
    return float((R @ C.T).max(axis=1).mean())  # best candidate match for each required skill

feat["skills_soft_coverage"] = [soft_coverage(c, r) for c, r in zip(res_skills, job_skills)]
print(f"{len(all_skills)} distinct skills encoded")

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

2863 distinct skills encoded


In [7]:
feat["candidate_experience_years"] = res["candidate_experience_years"]
feat["experience_required_years"] = job["experience_required_years"]
feat["experience_gap"] = feat["candidate_experience_years"] - feat["experience_required_years"]

# Candidate-side flags only (job-side flags are proxies for job identity)
feat["has_career_objective"] = raw["career_objective"].notna().astype(int)
feat["has_certifications"] = raw["certification_providers"].notna().astype(int)
feat["has_languages"] = raw["languages"].notna().astype(int)
feat["has_extra_curricular"] = raw["extra_curricular_activity_types"].notna().astype(int)

In [8]:
fallback = pd.Series("missing_resume_" + raw.index.astype(str), index=raw.index)
feat["resume_id"] = pd.factorize(raw["skills"].fillna(fallback))[0]
feat["job_id"] = pd.factorize(raw["job_position_name"].fillna("missing_job"))[0]
feat["job_position_name"] = raw["job_position_name"]   # metadata: NEVER a model input
feat["matched_score"] = raw["matched_score"]

In [9]:
model_features = [
    "semantic_similarity", "objective_responsibilities_similarity", "education_similarity",
    "skills_coverage", "skills_jaccard", "skills_soft_coverage",
    "candidate_experience_years", "experience_required_years", "experience_gap",
    "has_career_objective", "has_certifications", "has_languages", "has_extra_curricular",
]

print(feat[model_features].describe().T)
print("\n% de NaN :")
print(feat[model_features].isna().mean().round(3))
print("\nSpearman correlation with matched_score :")
print(feat[model_features + ["matched_score"]].corr(method="spearman")["matched_score"].sort_values())

                                        count       mean        std  \
semantic_similarity                    9544.0   0.259098   0.112958   
objective_responsibilities_similarity  4740.0   0.202939   0.096970   
education_similarity                   9460.0   0.436016   0.142330   
skills_coverage                        7751.0   0.014754   0.065676   
skills_jaccard                         7751.0   0.003548   0.016266   
skills_soft_coverage                   7751.0   0.342372   0.119297   
candidate_experience_years             8508.0  12.623868  11.256298   
experience_required_years              8180.0   3.541687   2.843748   
experience_gap                         7292.0   9.082525  11.611037   
has_career_objective                   9544.0   0.496647   0.500015   
has_certifications                     9544.0   0.210394   0.407610   
has_languages                          9544.0   0.073345   0.260715   
has_extra_curricular                   9544.0   0.358969   0.479723   

     

In [10]:
feat.to_parquet("../data/processed/v2_features.parquet")
print(feat.shape)

(9544, 17)


In [11]:
feat = pd.read_parquet("../data/processed/v2_features.parquet")

# 1) Share of score variance explained by job alone (the v1 shortcut)
job_mean = feat.groupby("job_id")["matched_score"].transform("mean")
ss_total = ((feat["matched_score"] - feat["matched_score"].mean()) ** 2).sum()
ss_within = ((feat["matched_score"] - job_mean) ** 2).sum()
print(f"Variance explained by job alone : {1 - ss_within / ss_total:.1%}")

# 2) Correlation WITHIN a job = ability to rank resumes for the same job posting
features_to_check = [
    "semantic_similarity", "objective_responsibilities_similarity", "education_similarity",
    "skills_coverage", "skills_soft_coverage", "candidate_experience_years",
    "experience_gap", "has_extra_curricular",
]
rows = []
for col in features_to_check:
    rs = []
    for _, g in feat.groupby("job_id"):
        sub = g[[col, "matched_score"]].dropna()
        if len(sub) >= 30 and sub[col].nunique() > 1:
            rs.append(sub[col].corr(sub["matched_score"], method="spearman"))
    rows.append({"feature": col, "within_job_spearman": np.nanmean(rs),
                 "std": np.nanstd(rs), "n_jobs": len(rs)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

Variance explained by job alone : 28.4%
                              feature  within_job_spearman   std  n_jobs
                  semantic_similarity                0.298 0.171      28
objective_responsibilities_similarity                0.176 0.113      28
                 education_similarity                0.222 0.144      28
                      skills_coverage                0.153 0.107      18
                 skills_soft_coverage                0.303 0.159      23
           candidate_experience_years                0.171 0.267      28
                       experience_gap                0.193 0.251      24
                 has_extra_curricular               -0.134 0.119      28
